In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import random


In [2]:
text = [
    "i love machine learning",
    "i love deep learning",
    "deep learning is fun",
    "machine learning is powerful",
]


In [3]:
tokens = [sentence.split() for sentence in text]


word_to_ix = {}
for sentence in tokens:
    for word in sentence:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)

ix_to_word = {v: k for k, v in word_to_ix.items()}
vocab_size = len(word_to_ix)

In [4]:
def prepare_data(tokens):
    sequences = []
    for sentence in tokens:
        for i in range(len(sentence) - 1):
            input_seq = sentence[:i+1]
            target = sentence[i+1]
            sequences.append((input_seq, target))
    return sequences

data = prepare_data(tokens)

def encode(seq):
    return torch.tensor([word_to_ix[w] for w in seq], dtype=torch.long)

In [5]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        embeds = self.embedding(x)
        lstm_out, _ = self.lstm(embeds.view(len(x), 1, -1))
        out = self.fc(lstm_out[-1])
        return out

In [6]:
model = RNNLanguageModel(vocab_size)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(200):
    total_loss = 0
    for seq, target in data:
        model.zero_grad()

        input_tensor = encode(seq)
        target_tensor = torch.tensor([word_to_ix[target]])

        output = model(input_tensor)
        loss = loss_fn(output, target_tensor)

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {total_loss:.4f}")

Epoch 0, Loss: 23.4917
Epoch 20, Loss: 1.5647
Epoch 40, Loss: 1.5180
Epoch 60, Loss: 1.4968
Epoch 80, Loss: 1.4821
Epoch 100, Loss: 1.4719
Epoch 120, Loss: 1.4660
Epoch 140, Loss: 1.4601
Epoch 160, Loss: 1.4561
Epoch 180, Loss: 1.4523


In [7]:
def generate_text(model, start_seq, max_len=5):
    model.eval()
    words = start_seq.split()

    for _ in range(max_len):
        input_tensor = encode(words)
        output = model(input_tensor)

        probs = torch.softmax(output, dim=1).detach().numpy()[0]
        next_word_ix = random.choices(range(vocab_size), weights=probs)[0]

        next_word = ix_to_word[next_word_ix]
        words.append(next_word)

    return " ".join(words)

In [8]:
print("\nGenerated Text:")
print(generate_text(model, "i love", max_len=5))
print(generate_text(model, "machine learning", max_len=5))


Generated Text:
i love deep learning is fun fun
machine learning is powerful powerful love deep
